In [1]:
import os
import xml.etree.ElementTree as ET

import pandas as pd

In [2]:
def is_img(pth):
    return os.path.splitext(pth)[-1].lower() in [".png", ".jpg", ".jpeg", ".bmp"]

In [19]:
def parse_text(txt):
    if txt == "true":
        return True
    elif txt == "false":
        return False
    else:
        return txt

def split_point(s):
    return s.split(";")[0].split(",")

def process_img_node(node: ET.Element, batch: str = None, folder: str = None) -> list:
    img_attributes = {
        "name": node.attrib["name"],
        "width": int(node.attrib["width"]),
        "height": int(node.attrib["height"])
    }

    boxes = []
    positions = []
    for child in node:
        attributes = {}
        for meta in child:
            attributes[meta.attrib["name"]] = parse_text(meta.text)

        obj = {
            **img_attributes,
            "label": child.attrib["label"],
            **attributes
        }
        if child.tag == "points":
            obj["pos"] = [float(i) for i in split_point(child.attrib["points"])]
            positions.append(obj)
        else:
            obj["box"] = [
                float(child.attrib["xtl"]),
                float(child.attrib["ytl"]),
                float(child.attrib["xbr"]),
                float(child.attrib["ybr"])
            ]
            boxes.append(obj)

    if len(boxes) != len(positions):
        print(f"[WARNING] Number of boxes not equal to number of positions: {len(boxes)} vs {len(positions)}. "
              f"Img: {img_attributes['name']}, folder: {folder}, batch: {batch}")

    objects = []
    for box in boxes:
        for pos in positions:
            if pos["ID"] == box["ID"]:
                box["pos"] = pos["pos"]
                objects.append(box)

    return objects

def process_annotation_file(pth: str, batch: str = None, folder: str = None) -> list:
    tree = ET.parse(pth)
    root = tree.getroot()
    objects = []
    for child in root:
        if child.tag != "image":
            continue
        objects += process_img_node(child, batch, folder)

    if batch is not None:
        for obj in objects:
            obj["batch"] = batch

    if folder is not None:
        for obj in objects:
            obj["folder"] = folder

    return objects

In [20]:
base_path = "../../../db/object-detection"
batches = sorted([i for i in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, i))])

folders = {}
for batch in batches:
    tmp_pth = os.path.join(base_path, batch)
    folders[batch] = sorted([i for i in os.listdir(tmp_pth) if os.path.isdir(os.path.join(tmp_pth, i))])

In [21]:
imgs = {}
annotations = {}
total_img_count = 0
for batch in batches:
    for folder in folders[batch]:
        tmp_pth = os.path.join(base_path, batch, folder)
        batch_imgs = [i for i in os.listdir(tmp_pth) if is_img(i)]
        batch_annotation = [i for i in os.listdir(tmp_pth) if i.endswith(".xml")][0]
        
        key = (batch, folder)
        imgs[key] = batch_imgs
        annotations[key] = batch_annotation
        total_img_count += len(batch_imgs)

print("total images:", total_img_count)

total images: 841


In [22]:
objects = []

for batch, annot in annotations.items():
    objects += process_annotation_file(os.path.join(base_path, *batch, annot), *batch)

df = pd.DataFrame(objects)

[WARNING] Number of boxes not equal to number of positions: 27 vs 26. Img: PG1_CIR_H270_office_HotDesk_13_10_12_41_2016.avi_snapshot_00.00.725.bmp, folder: 6, batch: batch4_22.5
[WARNING] Number of boxes not equal to number of positions: 35 vs 34. Img: PG1_CIR_H270_office_HotDesk_19_11_07_19_2016.avi_snapshot_00.02.019.bmp, folder: 6, batch: batch4_22.5


In [23]:
print("total number of images: ", df["name"].unique().shape[0])

print("total number of annotated objects: ", df.shape[0])

object_per_img = df.groupby("name").count()["width"].values
print(f"objects per img: min = {object_per_img.min()}. max = {object_per_img.max()}. avg = {object_per_img.mean():.02f}")

print("number of object per class:\n", df["label"].value_counts())

total number of images:  765
total number of annotated objects:  18787
objects per img: min = 1. max = 86. avg = 24.56
number of object per class:
 label
office_chair               5355
desk                       2581
desk_FULL                  1931
laptop_opened              1259
monitor                    1208
cabinet_FULL               1094
cabinet                     962
dinning_chair               535
keyboard                    525
plants                      503
bag                         449
coffee_table                409
laptop_closed               344
meeting_room_table          313
coffee_table_FULL           304
lounge_chair                251
whiteboard                  203
bar_stool                   195
coat                        152
sofa                        115
meeting_room_table_FULL      63
printer                      22
coffee_station                8
sink                          6
Name: count, dtype: int64


In [27]:
if df["name"].unique().shape[0] == total_img_count:
    print("OK")
elif df["name"].unique().shape[0] > total_img_count:
    tmp = []
    for i in imgs.values():
        tmp += i

    not_uploaded_imgs = set(df["name"].unique()).difference(tmp)
    print("total number of not uploaded images:", len(not_uploaded_imgs))
    print("batch | image")
    print("-" * 25)
    for i in not_uploaded_imgs:
        tmp = df[df["name"] == i][["name", "batch"]].iloc[0]
        print(f"{tmp['batch']:>5} | {tmp['name']}")
else:
    print("not annotated images")
    print("batch, folder             | image")
    print("-" * 25)
    for k, v in imgs.items():
        tmp = set(v).difference(df["name"].unique())
        for i in tmp:
            print(f"{str(k):<25} | {i}")
            
    print()
    print("duplicates")
    df1 = df.groupby(['name','batch', "folder"]).size().reset_index()[['name','batch', "folder"]]

    duplicates = df1[df1.duplicated('name', keep=False)]

    duplicate_info = duplicates.groupby('name').agg({
        'folder': lambda x: ', '.join(x),
        'batch': lambda x: ', '.join(x)
    }).reset_index()
    
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)

    print(duplicate_info)

not annotated images
batch, folder             | image
-------------------------
('batch3_21.5', '12')     | PG1_DL_H250_Walk_Beige_Carpet_17_09_34_45_2016-5_266.jpg
('batch4_22.5', '8')      | PG1_dev_H700_Walk_Q_Margalit_Mol_Q_1509273826-13_425.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_HD_NY_St170817_1499783863-16_1309.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_HD_NY_St170817_1499787515-13_81.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_HD_NY_St170817_1499784543-15_1283.jpg
('batch4_22.5', '8')      | PG1_dev_H700_Walk_Q_Margalit_Mol_Q_1509274161-16_142.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_HD_NY_St170817_1499785773-12_1524.jpg
('batch4_22.5', '8')      | PG1_dev_H700_Walk_Q_Margalit_Mol_Q_1509274765-20_619.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_HD_NY_St170817_1499787952-12_221.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_HD_NY_St170817_1499784084-16_1891.jpg
('batch4_22.5', '8')      | PG1_DL_H350_Office_MR_noDB_22_15_17_10_2017-

In [26]:
df.to_csv(os.path.join(base_path, "annotation_all.csv"), index=False)